# Task 2: Feature Engineering Challenge
Using a datetime-rich dataset (e.g., Uber rides, sales transactions, Airbnb listings), engineer at least 6 new features:
1. extract day-of-week, hour, is_weekend from timestamps
2. create 2 interaction features (multiply/ratio)
3. apply log transform to at least 1 skewed numeric column
4. bin one continuous variable
5. then compare model accuracy (use any simple classifier) before and after your engineered features
6. document the accuracy delta and explain why each feature help

### Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

### Load Dataset

In [2]:
df = pd.read_csv("retail_small.csv")

print(df.head())
print(df.shape)

  InvoiceNo StockCode                      Description  Quantity  \
0    540456     48185               DOORMAT FAIRY CAKE         2   
1    566891     23013    GLASS APOTHECARY BOTTLE TONIC         4   
2   C562139     21313      GLASS HEART T-LIGHT HOLDER         -4   
3    565438     22382       LUNCH BAG SPACEBOY DESIGN          4   
4    566016     21212  PACK OF 72 RETROSPOT CAKE CASES        24   

       InvoiceDate  UnitPrice  CustomerID         Country  
0   1/7/2011 12:14       7.95     13534.0  United Kingdom  
1  9/15/2011 13:51       3.95     14894.0  United Kingdom  
2   8/3/2011 10:10       0.85     12921.0  United Kingdom  
3   9/4/2011 13:56       1.65     17229.0  United Kingdom  
4   9/8/2011 12:20       0.55     15144.0  United Kingdom  
(10000, 8)


### Create Target Variable

Since the dataset has no target column, create one.

We will predict whether an order quantity is above the median.

In [3]:
median_qty = df["Quantity"].median()

df["HighQuantity"] = (
    df["Quantity"] > median_qty
).astype(int)

df["HighQuantity"].value_counts()

HighQuantity
0    5095
1    4905
Name: count, dtype: int64

### Convert Date Column

In [4]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

### Baseline Model (Before Feature Engineering)

Use only original features.

In [5]:
baseline_df = df.copy()

# Select features:

X_base = baseline_df[
    [
        "StockCode",
        "Description",
        "Country",
        "UnitPrice"
    ]
]

y = baseline_df["HighQuantity"]

### Train-Test Split

In [6]:
X_train_base, X_test_base, y_train, y_test = train_test_split(
    X_base,
    y,
    test_size=0.2,
    random_state=42
)

### Encode Categorical Features

In [7]:
cat_cols = [
    "StockCode",
    "Description",
    "Country"
]

num_cols = [
    "UnitPrice"
]

In [8]:
preprocessor_base = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            cat_cols
        ),
        (
            "num",
            "passthrough",
            num_cols
        )
    ]
)

### Build Baseline Model

In [9]:
baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor_base),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

In [10]:
# Train:

baseline_model.fit(X_train_base, y_train)

# Predict:

y_pred_base = baseline_model.predict(X_test_base)

# Accuracy:

baseline_accuracy = accuracy_score(
    y_test,
    y_pred_base
)

print("Baseline Accuracy:", baseline_accuracy)

Baseline Accuracy: 0.6625
